# BERT の出力から文ベクトルを自分で作る (mean pooling)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_transformers/mean_pooling.ipynb)

NLP クックブック 12-02 ／ Transformers の基本操作

Sentence Transformers を使わずに、モデルの隠れ状態から文の埋め込みを計算したい。パディングを正しく無視するのがポイント。

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

name = "intfloat/multilingual-e5-small"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModel.from_pretrained(name).eval()

sentences = ["query: 今日は天気が良いので散歩に行きます。", "query: 晴れた。歩こう。",
             "query: 明日の会議は午後 3 時から 5 階の大会議室で行いますので、資料を 10 部印刷して持ってきてください。"]
batch = tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
with torch.no_grad():
    hidden = model(**batch).last_hidden_state            # (文数, トークン数, 次元)
print(hidden.shape)

mask = batch["attention_mask"].unsqueeze(-1).float()      # パディング部分を 0 にするマスク
mean = (hidden * mask).sum(dim=1) / mask.sum(dim=1)       # 本物のトークンだけで平均
emb = F.normalize(mean, dim=-1)
print((emb @ emb.T).round(decimals=3))

torch.Size([3, 36, 384])
tensor([[1.0000, 0.8960, 0.8110],
        [0.8960, 1.0000, 0.7700],
        [0.8110, 0.7700, 1.0000]])


In [2]:
# よくある間違い: パディングも含めて平均してしまう / [CLS] を使う
naive = F.normalize(hidden.mean(dim=1), dim=-1)
cls = F.normalize(hidden[:, 0], dim=-1)
print("パディング込みの平均:", [round(x, 3) for x in (naive @ naive.T)[0].tolist()])
print("[CLS] ベクトル:      ", [round(x, 3) for x in (cls @ cls.T)[0].tolist()])

from sentence_transformers import SentenceTransformer
st = torch.tensor(SentenceTransformer(name).encode(sentences, normalize_embeddings=True))
print("Sentence Transformers と一致:", torch.allclose(st, emb, atol=1e-4))

パディング込みの平均: [1.0, 0.92, 0.772]
[CLS] ベクトル:       [1.0, 0.942, 0.888]
Sentence Transformers と一致: True


- どのプーリング (平均 / CLS / 最後のトークン) を使うべきかはモデルの学習方法で決まる。モデルカードの指定に従う。
- 素の BERT (文埋め込み用に学習されていないモデル) の平均ベクトルは、類似度計算にはあまり向かない。